In [1]:
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d
import os

In [11]:
def center_and_scale_landmarks(data):
    """
    Centers and scales facial landmarks using the mean and standard deviation.

    Parameters:
        data (pd.DataFrame): Input DataFrame with columns ['frame', 'x', 'y'].

    Returns:
        pd.DataFrame: Transformed DataFrame with centered and scaled coordinates.
    """
    # Create a copy of the data to avoid modifying the original DataFrame
    data = data.copy()

    # Calculate mean and standard deviation of x and y for each frame
    grouped = data.groupby("frame")
    means = grouped[['x', 'y']].mean()
    stds = grouped[['x', 'y']].std()

    # Merge mean and std back to the original DataFrame
    data = data.merge(means, on="frame", suffixes=("", "_mean"))
    data = data.merge(stds, on="frame", suffixes=("", "_std"))

    # Center and scale the coordinates
    data['x_scaled'] = (data['x'] - data['x_mean']) / data['x_std']
    data['y_scaled'] = (data['y'] - data['y_mean']) / data['y_std']

    # Drop intermediate columns (optional)
    data = data.drop(columns=['x', 'y', 'x_mean', 'y_mean', 'x_std', 'y_std'])

    return data


In [12]:
# Function to interpolate the landmark coordinates
def interpolate_landmarks(df, original_fps, target_fps):
    # Create an empty list to store the interpolated results
    interpolated_data = []

    # Get the unique landmark numbers
    landmark_numbers = df['landmark_index'].unique()
    
    # Iterate over each landmark number
    for landmark in landmark_numbers:
        # Get the rows for the current landmark
        landmark_data = df[df['landmark_index'] == landmark]
        
        # Get the frame numbers and coordinates
        frames = landmark_data['frame'].values
        x_coords = landmark_data['x_scaled'].values
        y_coords = landmark_data['y_scaled'].values
        
        # Create interpolation functions for x and y coordinates
        interp_x = interp1d(frames, x_coords, kind='linear', fill_value="extrapolate")
        interp_y = interp1d(frames, y_coords, kind='linear', fill_value="extrapolate")
        
        # Generate the new frame numbers based on the target fps
        new_frame_numbers = np.arange(0, int(frames[-1] * (target_fps / original_fps)) + 1)

        # Calculate the interpolated coordinates for the new frame numbers
        new_x_coords = interp_x(new_frame_numbers)
        new_y_coords = interp_y(new_frame_numbers)
        
        # Add the new data to the interpolated list
        for new_frame, new_x, new_y in zip(new_frame_numbers, new_x_coords, new_y_coords):
            interpolated_data.append({
                'frame': new_frame,
                'landmark_index': landmark,
                'x_scaled': new_x,
                'y_scaled': new_y
            })

    # Convert the interpolated data to a DataFrame
    interpolated_df = pd.DataFrame(interpolated_data)
    
    return interpolated_df


In [13]:
def process_videos(input_dir, output_dir, video_info_path, min_duration=500):
    """
    Processes all video landmark files in the input directory, applying scaling and interpolation.

    Parameters:
        input_dir (str): Path to the input directory containing landmark files.
        output_dir (str): Path to the output directory to save processed files.
        video_info_path (str): Path to the CSV file containing video information with FPS.
    """
    print("Loading video information...")
    video_info = pd.read_csv(video_info_path)
    #target_fps = int(video_info['fps'].mean())
    target_fps = 27
    print(f"Target FPS set to {target_fps}")

    print("Starting to process video files...")
    for file_name in os.listdir(input_dir):
        if file_name.endswith(".csv"):
            video_name = file_name.split(".")[0]
            print("---------------")
            print(f"Processing file: {file_name}")
            video_data = pd.read_csv(os.path.join(input_dir, file_name))

            # Get original FPS
            original_fps = video_info[video_info['video_name'] == video_name]['fps'].values[0]
            print(f"Original FPS for {video_name}: {original_fps}")

            # Calculate video duration
            video_duration = video_info[video_info['video_name'] == video_name]['duration'].values[0]
            print(f"Video duration: {video_duration:.2f} seconds")

            # Skip videos shorter than 600 seconds
            if video_duration <= (min_duration+1):
                print(f"Skipping {video_name} as it is shorter than 600 seconds.")
                continue

            # Take the first 600 * specific_frame_number frames
            max_frames = (min_duration+1) * original_fps
            video_data = video_data[video_data['frame'] < max_frames]
            print(f"Retained first {max_frames} frames.")

            # Apply transformations
            print(f"Scaling landmarks for {video_name}...")
            scaled_data = center_and_scale_landmarks(video_data)
            print(f"Interpolating landmarks for {video_name}...")
            interpolated_data = interpolate_landmarks(scaled_data, original_fps, target_fps)
            
            interpolated_data = interpolated_data[interpolated_data['frame']<=target_fps*min_duration]
            # Ensure output directory exists
            os.makedirs(output_dir, exist_ok=True)

            # Save the processed data with the same name
            output_path = os.path.join(output_dir, file_name)
            interpolated_data.to_csv(output_path, index=False)
            print(f"Saved processed data to {output_path}")

    print("Processing complete!")


In [14]:
input_directory = "Data/face_landmarks/F4/Raw/"  # Replace with your input directory path
output_directory = "Data/face_landmarks/F4/Preprocessed/"  # Replace with your output directory path
video_info_csv = "Data/video_info.csv"  # Replace with your video info CSV path

process_videos(input_directory, output_directory, video_info_csv)

Loading video information...
Target FPS set to 27
Starting to process video files...
---------------
Processing file: 4_37_0.csv
Original FPS for 4_37_0: 29.88037928519329
Video duration: 685.50 seconds
Retained first 14970.070021881838 frames.
Scaling landmarks for 4_37_0...
Interpolating landmarks for 4_37_0...
Saved processed data to Data/face_landmarks/F4/Preprocessed/4_37_0.csv
---------------
Processing file: 4_37_10.csv
Original FPS for 4_37_10: 29.86674476497291
Video duration: 682.90 seconds
Retained first 14963.239127251429 frames.
Scaling landmarks for 4_37_10...
Interpolating landmarks for 4_37_10...
Saved processed data to Data/face_landmarks/F4/Preprocessed/4_37_10.csv
---------------
Processing file: 4_40_0.csv
Original FPS for 4_40_0: 29.970874565387373
Video duration: 602.59 seconds
Retained first 15015.408157259073 frames.
Scaling landmarks for 4_40_0...
Interpolating landmarks for 4_40_0...
Saved processed data to Data/face_landmarks/F4/Preprocessed/4_40_0.csv
------

In [2]:
def calculate_video_metrics_with_tracking(data, blink_threshold=0.2, yawn_threshold=0.1):
    """
    Calculate various metrics from facial landmark data with progress tracking.

    Parameters:
        data (pd.DataFrame): DataFrame with columns ['frame', 'landmark_index', 'x_scaled', 'y_scaled'].

    Returns:
        dict: A dictionary containing aggregated metrics for the video.
    """
    print("Starting metric calculation process.")

    metrics = {
        "blink_count": 0,
        "mean_blink_duration": 0,
        "min_blink_duration": float('inf'),
        "max_blink_duration": 0,
        "std_blink_duration": 0,
        "mean_left_eye_openness": 0,
        "mean_right_eye_openness": 0,
        "average_smile_intensity": 0, 
        "max_smile_intensity": 0,
        "total_horizontal_movement": 0, 
        "total_vertical_movement": 0,
        "yawn_count": 0,
        "mean_yawn_duration": 0,
        "max_yawn_duration": 0,
        "std_yawn_duration": 0,        
        "y_tilt_mean": 0, 
        "y_tilt_max": 0, 
        "y_tilt_std": 0,
        "x_tilt_mean": 0, 
        "x_tilt_max": 0, 
        "x_tilt_std": 0,
    }

    left_eye_upper_indices = [466, 388, 387, 386, 385, 384, 398]
    left_eye_lower_indices = [263, 249, 390, 373, 374, 380, 381, 382, 362]
    left_eye_indices = left_eye_upper_indices + left_eye_lower_indices
    right_eye_upper_indices = [246, 161, 160, 159, 158, 157, 173]
    right_eye_lower_indices = [33, 7, 163, 144, 145, 153, 154, 155, 133]
    right_eye_indices = right_eye_upper_indices + right_eye_lower_indices
    lips_upper_outer= [61, 185, 40, 39, 37, 0, 267, 269, 270, 409, 291]
    lips_lower_outer= [146, 91, 181, 84, 17, 314, 405, 321, 375, 291]
    lips_upper_inner= [78, 191, 80, 81, 82, 13, 312, 311, 310, 415, 308]
    lips_lower_inner= [78, 95, 88, 178, 87, 14, 317, 402, 318, 324, 308]
    middle_line_indices= [10,9,6,4,1,152]
    nose_tip_index = 1


    def eye_aspect_ratio(eye_landmarks):
        # Compute distances between the vertical landmarks

        vertical_1 = np.linalg.norm(
            eye_landmarks.iloc[5][['x_scaled', 'y_scaled']].values - 
            eye_landmarks.iloc[9][['x_scaled', 'y_scaled']].values
        )
        vertical_2 = np.linalg.norm(
            eye_landmarks.iloc[3][['x_scaled', 'y_scaled']].values - 
            eye_landmarks.iloc[11][['x_scaled', 'y_scaled']].values
        )
        # Compute distance between the horizontal landmarks
        horizontal = np.linalg.norm(
            eye_landmarks.iloc[1][['x_scaled', 'y_scaled']].values - 
            eye_landmarks.iloc[2][['x_scaled', 'y_scaled']].values
        )

        return (vertical_1 + vertical_2) / (2.0 * horizontal)

    def compute_openness(eye_landmarks):
        vertical_1 = np.linalg.norm(
            eye_landmarks.iloc[5][['x_scaled', 'y_scaled']].values -
            eye_landmarks.iloc[9][['x_scaled', 'y_scaled']].values
        )
        vertical_2 = np.linalg.norm(
            eye_landmarks.iloc[3][['x_scaled', 'y_scaled']].values -
            eye_landmarks.iloc[11][['x_scaled', 'y_scaled']].values
        )
        return (vertical_1 + vertical_2) / 2

    def mouth_aspect_ratio(upper_lip_landmarks, lower_lip_landmarks):
        # Calculate the vertical distance between the upper and lower lip
        vertical_dist = np.linalg.norm(
            upper_lip_landmarks[['x_scaled', 'y_scaled']].mean().values -
            lower_lip_landmarks[['x_scaled', 'y_scaled']].mean().values
        )
        return vertical_dist


    blink_durations = []
    blink_timestamps = []
    is_blinking = False
    blink_start = None
    smile_intensities = []
    yawn_timestamps = []
    yawn_durations = []
    is_yawning = False
    yawn_start = None
    tilts_y = []
    tilts_x = []
    
    
    for frame, frame_data in data.groupby('frame'):
        if frame % (60*27) == 0:
            print(f"{frame} minute {frame/(60*27)}")
        
        eye_openness_per_frame = {"left_eye_openness": [], "right_eye_openness": []}
        
        left_eye = frame_data[frame_data['landmark_index'].isin(left_eye_indices)]
        right_eye = frame_data[frame_data['landmark_index'].isin(right_eye_indices)]
        upper_outer = frame_data[frame_data['landmark_index'].isin(lips_upper_outer)]
        lower_outer = frame_data[frame_data['landmark_index'].isin(lips_lower_outer)]
        upper_inner = frame_data[frame_data['landmark_index'].isin(lips_upper_inner)]
        lower_inner = frame_data[frame_data['landmark_index'].isin(lips_lower_inner)]
        y_coords = frame_data[frame_data['landmark_index'].isin(middle_line_indices)]['y_scaled'].values
        x_coords = frame_data[frame_data['landmark_index'].isin(middle_line_indices)]['x_scaled'].values
        
        if not left_eye.empty and not right_eye.empty:
            left_ear = eye_aspect_ratio(left_eye)
            right_ear = eye_aspect_ratio(right_eye)
            mean_ear = (left_ear + right_ear) / 2
            
            if mean_ear < blink_threshold:
                if not is_blinking:
                    is_blinking = True
                    blink_start = frame
            else:
                if is_blinking:
                    is_blinking = False
                    blink_durations.append(frame - blink_start)
                    blink_timestamps.append(frame/27)
        
                    
                    
        # Calculate openness if data for both eyes is available
        if not left_eye.empty:
            left_openness = compute_openness(left_eye)
            eye_openness_per_frame["left_eye_openness"].append(left_openness)
        else:
            eye_openness_per_frame["left_eye_openness"].append(None)

        if not right_eye.empty:
            right_openness = compute_openness(right_eye)
            eye_openness_per_frame["right_eye_openness"].append(right_openness)
        else:
            eye_openness_per_frame["right_eye_openness"].append(None)
            
            
            
        if not (upper_outer.empty or lower_outer.empty or upper_inner.empty or lower_inner.empty):
            # Horizontal distance (width of the smile)
            left_corner = frame_data[frame_data['landmark_index'] == lips_upper_outer[0]][['x_scaled', 'y_scaled']].values
            right_corner = frame_data[frame_data['landmark_index'] == lips_upper_outer[-1]][['x_scaled', 'y_scaled']].values
            if left_corner.size > 0 and right_corner.size > 0:
                horizontal_distance = np.linalg.norm(left_corner - right_corner)

            # Vertical distance (openness of the mouth)
            upper_mean = upper_inner[['x_scaled', 'y_scaled']].mean().values
            lower_mean = lower_inner[['x_scaled', 'y_scaled']].mean().values
            vertical_distance = np.linalg.norm(upper_mean - lower_mean)

            # Combined smile intensity
            smile_intensity = horizontal_distance + vertical_distance
            smile_intensities.append(smile_intensity)
            
        
        if not upper_outer.empty and not lower_outer.empty:
            mar = mouth_aspect_ratio(upper_outer, lower_outer)
            
            if mar > yawn_threshold:
                if not is_yawning:
                    is_yawning = True
                    yawn_start = frame
            else:
                if is_yawning:
                    is_yawning = False
                    yawn_durations.append(frame - yawn_start)
                    yawn_timestamps.append(frame/27)
                    
                    
        # Calculate y-axis tilt (vertical spread)
        if len(y_coords) > 1:
            y_tilt = max(y_coords) - min(y_coords)
            tilts_y.append(y_tilt)

        # Calculate x-axis tilt (horizontal spread)
        if len(x_coords) > 1:
            x_tilt = max(x_coords) - min(x_coords)
            tilts_x.append(x_tilt)
    
    
    if blink_durations:
        metrics["blink_count"] = len(blink_durations)
        metrics["mean_blink_duration"] = np.mean(blink_durations)
        metrics["min_blink_duration"] = np.min(blink_durations)
        metrics["max_blink_duration"] = np.max(blink_durations)
        metrics["std_blink_duration"] = np.std(blink_durations)

    
    metrics["mean_left_eye_openness"] = np.nanmean(eye_openness_per_frame["left_eye_openness"])
    metrics["mean_right_eye_openness"] = np.nanmean(eye_openness_per_frame["right_eye_openness"])

    if smile_intensities:
        metrics["average_smile_intensity"] = np.mean(smile_intensities)
        metrics["max_smile_intensity"] = np.max(smile_intensities)



    nose_data = data[data['landmark_index'] == nose_tip_index].sort_values('frame')

    if not nose_data.empty:
        # Calculate frame-to-frame differences
        horizontal_diff = np.abs(np.diff(nose_data['x_scaled'], prepend=nose_data['x_scaled'].iloc[0]))
        vertical_diff = np.abs(np.diff(nose_data['y_scaled'], prepend=nose_data['y_scaled'].iloc[0]))

        # Sum up the movements
        metrics["total_horizontal_movement"] = np.sum(horizontal_diff)
        metrics["total_vertical_movement"] = np.sum(vertical_diff)
        

    if yawn_durations:
        metrics["yawn_count"] = len(yawn_durations)
        metrics["mean_yawn_duration"] = np.mean(yawn_durations)
        metrics["max_yawn_duration"] = np.max(yawn_durations)
        metrics["std_yawn_duration"] = np.std(yawn_durations)


    # Calculate metrics for y-axis tilt
    if tilts_y:
        metrics["y_tilt_mean"] = np.mean(tilts_y)
        metrics["y_tilt_max"] = np.max(tilts_y)
        metrics["y_tilt_std"] = np.std(tilts_y)

    # Calculate metrics for x-axis tilt
    if tilts_x:
        metrics["x_tilt_mean"] = np.mean(tilts_x)
        metrics["x_tilt_max"] = np.max(tilts_x)
        metrics["x_tilt_std"] = np.std(tilts_x)
    

    return metrics


In [3]:
import os
import pandas as pd
from multiprocessing import Pool

def process_single_file(args):
    filename, input_folder, output_folder, blink_threshold, yawn_threshold = args

    input_file = os.path.join(input_folder, filename)
    output_file = os.path.join(output_folder, f"{os.path.splitext(filename)[0]}_metrics.csv")

    # Read the CSV file into a DataFrame
    try:
        data = pd.read_csv(input_file)
    except Exception as e:
        print(f"Error reading {filename}: {e}")
        return

    # Call the function to calculate metrics
    metrics = calculate_video_metrics_with_tracking(data, blink_threshold, yawn_threshold)

    # Save the output metrics to a new CSV file in the output folder
    try:
        metrics_df = pd.DataFrame([metrics])
        metrics_df.to_csv(output_file, index=False)
        print(f"Processed {filename} and saved metrics to {output_file}")
    except Exception as e:
        print(f"Error saving metrics for {filename}: {e}")

def process_batch_of_files(batch_files, input_folder, output_folder, blink_threshold, yawn_threshold, num_processes):
    # Prepare arguments for parallel processing
    args = [
        (filename, input_folder, output_folder, blink_threshold, yawn_threshold)
        for filename in batch_files
    ]

    # Use multiprocessing Pool for parallel processing
    with Pool(processes=num_processes) as pool:
        pool.map(process_single_file, args)

def process_all_csv_files(input_folder, output_folder, blink_threshold=0.2, yawn_threshold=0.1, num_processes=4, batch_size=10):
    # Ensure output folder exists
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Collect all CSV files in the input folder
    csv_files = [f for f in os.listdir(input_folder) if f.endswith(".csv")]

    # Process files in batches
    for i in range(0, len(csv_files), batch_size):
        batch_files = csv_files[i:i + batch_size]
        print(f"Processing batch {i // batch_size + 1}: {batch_files}")
        process_batch_of_files(batch_files, input_folder, output_folder, blink_threshold, yawn_threshold, num_processes)


In [4]:
# Example usage:
input_folder = 'Data/face_landmarks/F2/Preprocessed/'
output_folder = 'Data/face_landmarks/F2/Summary/'
process_all_csv_files(input_folder, output_folder, num_processes=10, batch_size=10)

Processing batch 1: ['2_13_0.csv', '2_13_10.csv', '2_13_5.csv', '2_15_10.csv', '2_14_5.csv', '2_15_5.csv', '2_17_0.csv', '2_17_10.csv', '2_17_5.csv', '2_18_0.csv']
Starting metric calculation process.
Starting metric calculation process.
Starting metric calculation process.
Starting metric calculation process.
Starting metric calculation process.
Starting metric calculation process.
Starting metric calculation process.
Starting metric calculation process.
Starting metric calculation process.
0 minute 0.0
0 minute 0.0
0 minute 0.0
0 minute 0.0
0 minute 0.0
Starting metric calculation process.
0 minute 0.0
0 minute 0.0
0 minute 0.0
0 minute 0.0
0 minute 0.0
1620 minute 1.0
1620 minute 1.0
1620 minute 1.0
1620 minute 1.0
1620 minute 1.0
1620 minute 1.0
1620 minute 1.0
1620 minute 1.0
1620 minute 1.0
1620 minute 1.0
3240 minute 2.0
3240 minute 2.0
3240 minute 2.0
3240 minute 2.0
3240 minute 2.0
3240 minute 2.0
3240 minute 2.0
3240 minute 2.0
3240 minute 2.0
3240 minute 2.0
4860 minute 3.0


In [ ]:
import os
import pandas as pd

def process_all_csv_files(input_folder, output_folder, blink_threshold=0.2, yawn_threshold=0.1):
    # Ensure output folder exists
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    # Loop through all CSV files in the input folder
    for filename in os.listdir(input_folder):
        if filename.endswith(".csv"):
            # Construct the full file path
            input_file = os.path.join(input_folder, filename)
            
            # Read the CSV file into a DataFrame
            try:
                data = pd.read_csv(input_file)
            except Exception as e:
                print(f"Error reading {filename}: {e}")
                continue
            
            # Call the function to calculate metrics
            metrics = calculate_video_metrics_with_tracking(data, blink_threshold, yawn_threshold)
            
            # Save the output metrics to a new CSV file in the output folder
            output_file = os.path.join(output_folder, f"{os.path.splitext(filename)[0]}_metrics.csv")
            try:
                # Convert the metrics dictionary to a DataFrame and save to CSV
                metrics_df = pd.DataFrame([metrics])
                metrics_df.to_csv(output_file, index=False)
                print(f"Processed {filename} and saved metrics to {output_file}")
            except Exception as e:
                print(f"Error saving metrics for {filename}: {e}")

# Example usage:
input_folder = 'Data/face_landmarks/F4/Preprocessed/'
output_folder = 'Data/face_landmarks/F4/Summary/'

process_all_csv_files(input_folder, output_folder)


Starting metric calculation process.
0 minute 0.0
1620 minute 1.0
3240 minute 2.0
4860 minute 3.0
6480 minute 4.0
8100 minute 5.0
9720 minute 6.0
11340 minute 7.0
12960 minute 8.0
Processed 4_37_0.csv and saved metrics to Data/face_landmarks/F4/Summary/4_37_0_metrics.csv
Starting metric calculation process.
0 minute 0.0
1620 minute 1.0
3240 minute 2.0
4860 minute 3.0
6480 minute 4.0
8100 minute 5.0
9720 minute 6.0
11340 minute 7.0
12960 minute 8.0
Processed 4_37_10.csv and saved metrics to Data/face_landmarks/F4/Summary/4_37_10_metrics.csv
Starting metric calculation process.
0 minute 0.0
1620 minute 1.0
3240 minute 2.0
4860 minute 3.0
6480 minute 4.0
8100 minute 5.0
9720 minute 6.0
11340 minute 7.0
12960 minute 8.0
Processed 4_40_0.csv and saved metrics to Data/face_landmarks/F4/Summary/4_40_0_metrics.csv
Starting metric calculation process.
0 minute 0.0
1620 minute 1.0
3240 minute 2.0
4860 minute 3.0
6480 minute 4.0
8100 minute 5.0
9720 minute 6.0
11340 minute 7.0
12960 minute 8.0
Pr

In [53]:
nose_data = data[data['landmark_index'] == 1].sort_values('frame')

if not nose_data.empty:
    # Calculate frame-to-frame differences
    horizontal_diff = np.abs(np.diff(nose_data['x_scaled'], prepend=nose_data['x_scaled'].iloc[0]))
    vertical_diff = np.abs(np.diff(nose_data['y_scaled'], prepend=nose_data['y_scaled'].iloc[0]))


In [54]:
np.sum(horizontal_diff)

109.45951008188739

In [35]:
metrics = calculate_video_metrics_with_tracking(data)

Starting metric calculation process.
0 minute 0.0
1620 minute 1.0
3240 minute 2.0
4860 minute 3.0
6480 minute 4.0
8100 minute 5.0
9720 minute 6.0
11340 minute 7.0
12960 minute 8.0


In [36]:
metrics

{'blink_count': 136,
 'mean_blink_duration': 2.286764705882353,
 'min_blink_duration': 1,
 'max_blink_duration': 26,
 'std_blink_duration': 2.889911682538408,
 'mean_left_eye_openness': (0.19556107294809869,),
 'mean_right_eye_openness': (0.21406193331826165,),
 'average_smile_intensity': 1.4963557863906305,
 'max_smile_intensity': 1.7086868985674104,
 'total_horizontal_movement': 109.45951008188739,
 'total_vertical_movement': 79.60740305838294,
 'yawn_count': 0,
 'mean_yawn_duration': 0,
 'max_yawn_duration': 0,
 'std_yawn_duration': 0,
 'y_tilt_mean': 4.592226048292161,
 'y_tilt_max': 4.816677614937163,
 'y_tilt_std': 0.03536946896574072,
 'x_tilt_mean': 0.37785153455562864,
 'x_tilt_max': 2.415456985988463,
 'x_tilt_std': 0.10292361733892635}

In [40]:
def calculate_blinks(df, left_eye_indices, right_eye_indices, blink_threshold=0.2):
    """
    Calculate the number of blinks and the duration of blinks based on eye aspect ratio.

    Parameters:
        df (pd.DataFrame): DataFrame containing frame, landmark_index, x_scaled, y_scaled.
        left_eye_indices (list): List of indices for the left eye (upper and lower).
        right_eye_indices (list): List of indices for the right eye (upper and lower).
        blink_threshold (float): Threshold for detecting a blink.

    Returns:
        dict: Contains blink metrics like count, mean, min, max, and std duration.
    """
    print("Calculating blinks...")
    blink_metrics = {
        "blink_count": 0,
        "mean_blink_duration": 0,
        "min_blink_duration": float('inf'),
        "max_blink_duration": 0,
        "std_blink_duration": 0
    }
    
    def eye_aspect_ratio(eye_landmarks):
        # Compute distances between the vertical landmarks

        vertical_1 = np.linalg.norm(
            eye_landmarks.iloc[5][['x_scaled', 'y_scaled']].values - 
            eye_landmarks.iloc[9][['x_scaled', 'y_scaled']].values
        )
        vertical_2 = np.linalg.norm(
            eye_landmarks.iloc[3][['x_scaled', 'y_scaled']].values - 
            eye_landmarks.iloc[11][['x_scaled', 'y_scaled']].values
        )
        # Compute distance between the horizontal landmarks
        horizontal = np.linalg.norm(
            eye_landmarks.iloc[1][['x_scaled', 'y_scaled']].values - 
            eye_landmarks.iloc[2][['x_scaled', 'y_scaled']].values
        )

        return (vertical_1 + vertical_2) / (2.0 * horizontal)

    # Combine left and right eye indices
    all_eye_indices = left_eye_indices + right_eye_indices
    
    # Filter for only eye landmarks
    eye_data = df[df['landmark_index'].isin(all_eye_indices)]
    
    blink_durations = []
    blink_timestamps = []
    is_blinking = False
    blink_start = None
    
    for frame, frame_data in eye_data.groupby('frame'):
        if frame % (60*27) == 0:
            print(f"{frame} minute {frame/(60*27)}")
            
        left_eye = frame_data[frame_data['landmark_index'].isin(left_eye_indices)]
        right_eye = frame_data[frame_data['landmark_index'].isin(right_eye_indices)]
        
        if not left_eye.empty and not right_eye.empty:
            left_ear = eye_aspect_ratio(left_eye)
            right_ear = eye_aspect_ratio(right_eye)
            mean_ear = (left_ear + right_ear) / 2
            
            if mean_ear < blink_threshold:
                if not is_blinking:
                    is_blinking = True
                    blink_start = frame
            else:
                if is_blinking:
                    is_blinking = False
                    blink_durations.append(frame - blink_start)
                    blink_timestamps.append(frame/27)
                    print(f"blinked {frame/27}")
    
    if blink_durations:
        blink_metrics["blink_count"] = len(blink_durations)
        blink_metrics["mean_blink_duration"] = np.mean(blink_durations)
        blink_metrics["min_blink_duration"] = np.min(blink_durations)
        blink_metrics["max_blink_duration"] = np.max(blink_durations)
        blink_metrics["std_blink_duration"] = np.std(blink_durations)
    
    print("Blinks calculated.")
    return blink_metrics

In [47]:
left_eye_upper_indices = [466, 388, 387, 386, 385, 384, 398]
left_eye_lower_indices = [263, 249, 390, 373, 374, 380, 381, 382, 362]
right_eye_upper_indices = [246, 161, 160, 159, 158, 157, 173]
right_eye_lower_indices = [33, 7, 163, 144, 145, 153, 154, 155, 133]
blink_metrics = calculate_blinks(data, left_eye_upper_indices + left_eye_lower_indices, right_eye_upper_indices + right_eye_lower_indices)

Calculating blinks...
0 minute 0.0
blinked 0.1111111111111111
blinked 1.2592592592592593
blinked 7.444444444444445
blinked 12.185185185185185
blinked 17.37037037037037
blinked 22.0
blinked 24.51851851851852
blinked 34.48148148148148
blinked 36.03703703703704
blinked 56.55555555555556
1620 minute 1.0
blinked 65.77777777777777
blinked 81.03703703703704
blinked 86.88888888888889
blinked 91.88888888888889
blinked 107.88888888888889
3240 minute 2.0
blinked 133.0
blinked 134.0
blinked 138.44444444444446
blinked 149.92592592592592
blinked 171.22222222222223
4860 minute 3.0
blinked 181.77777777777777
blinked 194.2962962962963
blinked 215.25925925925927
6480 minute 4.0
blinked 240.37037037037038
blinked 249.7037037037037
blinked 258.9259259259259
blinked 267.48148148148147
blinked 278.74074074074076
blinked 280.6296296296296
blinked 291.22222222222223
blinked 299.037037037037
8100 minute 5.0
blinked 312.4074074074074
blinked 316.3703703703704
blinked 324.5925925925926
blinked 328.81481481481484

In [43]:
blink_metrics

{'blink_count': 136,
 'mean_blink_duration': 2.286764705882353,
 'min_blink_duration': 1,
 'max_blink_duration': 26,
 'std_blink_duration': 2.889911682538408}

In [60]:
def calculate_eye_openness(df, left_eye_indices, right_eye_indices):
    """
    Calculate the average eye openness over frames.

    Parameters:
        df (pd.DataFrame): DataFrame containing frame, landmark_index, x_scaled, y_scaled.
        left_eye_indices (list): List of indices for the left eye (upper and lower).
        right_eye_indices (list): List of indices for the right eye (upper and lower).

    Returns:
        dict: Contains average eye openness metrics for both eyes.
    """
    print("Calculating eye openness...")
    eye_openness_per_frame = {"left_eye_openness": [], "right_eye_openness": []}

    def compute_openness(eye_landmarks):
        vertical_1 = np.linalg.norm(
            eye_landmarks.iloc[5][['x_scaled', 'y_scaled']].values -
            eye_landmarks.iloc[9][['x_scaled', 'y_scaled']].values
        )
        vertical_2 = np.linalg.norm(
            eye_landmarks.iloc[3][['x_scaled', 'y_scaled']].values -
            eye_landmarks.iloc[11][['x_scaled', 'y_scaled']].values
        )
        return (vertical_1 + vertical_2) / 2

    # Combine left and right eye indices
    all_eye_indices = left_eye_indices + right_eye_indices

    # Filter for only eye landmarks
    eye_data = df[df['landmark_index'].isin(all_eye_indices)]

    for frame, frame_data in eye_data.groupby('frame'):
        if frame % (60 * 27) == 0:
            print(f"{frame} frames processed, corresponding to {frame / (60 * 27):.2f} minutes.")
        
        # Get left and right eye data for the current frame
        left_eye = frame_data[frame_data['landmark_index'].isin(left_eye_indices)]
        
        right_eye = frame_data[frame_data['landmark_index'].isin(right_eye_indices)]

        # Calculate openness if data for both eyes is available
        if not left_eye.empty:
            left_openness = compute_openness(left_eye)
            eye_openness_per_frame["left_eye_openness"].append(left_openness)
        else:
            eye_openness_per_frame["left_eye_openness"].append(None)

        if not right_eye.empty:
            right_openness = compute_openness(right_eye)
            eye_openness_per_frame["right_eye_openness"].append(right_openness)
        else:
            eye_openness_per_frame["right_eye_openness"].append(None)

    # Calculate mean openness for both eyes
    mean_left_openness = np.nanmean(eye_openness_per_frame["left_eye_openness"])
    mean_right_openness = np.nanmean(eye_openness_per_frame["right_eye_openness"])

    print("Eye openness calculated.")
    return {
        "mean_left_eye_openness": mean_left_openness,
        "mean_right_eye_openness": mean_right_openness,
    }


In [48]:
data.shape

(6318468, 4)

In [61]:
left_eye_upper_indices = [466, 388, 387, 386, 385, 384, 398]
left_eye_lower_indices = [263, 249, 390, 373, 374, 380, 381, 382, 362]
right_eye_upper_indices = [246, 161, 160, 159, 158, 157, 173]
right_eye_lower_indices = [33, 7, 163, 144, 145, 153, 154, 155, 133]
openness = calculate_eye_openness(data, left_eye_upper_indices + left_eye_lower_indices, right_eye_upper_indices + right_eye_lower_indices)
openness

Calculating eye openness...
0 frames processed, corresponding to 0.00 minutes.
1620 frames processed, corresponding to 1.00 minutes.
3240 frames processed, corresponding to 2.00 minutes.
4860 frames processed, corresponding to 3.00 minutes.
6480 frames processed, corresponding to 4.00 minutes.
8100 frames processed, corresponding to 5.00 minutes.
9720 frames processed, corresponding to 6.00 minutes.
11340 frames processed, corresponding to 7.00 minutes.
12960 frames processed, corresponding to 8.00 minutes.
Eye openness calculated.


{'mean_left_eye_openness': 0.1892796393161288,
 'mean_right_eye_openness': 0.2031275956228068}

In [3]:
def calculate_smiles(df, lips_upper_outer, lips_lower_outer, lips_upper_inner, lips_lower_inner):
    """
    Calculate smile intensity based on the horizontal stretch and vertical openness of the mouth.

    Parameters:
        df (pd.DataFrame): DataFrame containing frame, landmark_index, x_scaled, y_scaled.
        lips_upper_outer (list): Indices for the outer upper lip.
        lips_lower_outer (list): Indices for the outer lower lip.
        lips_upper_inner (list): Indices for the inner upper lip.
        lips_lower_inner (list): Indices for the inner lower lip.

    Returns:
        dict: Contains average smile intensity and max smile intensity.
    """
    print("Calculating smiles...")
    smile_metrics = {"average_smile_intensity": 0, "max_smile_intensity": 0}
    smile_intensities = []

    for frame, frame_data in df.groupby('frame'):
        if frame % (60 * 27) == 0:
            print(f"{frame} frames processed, corresponding to {frame / (60 * 27):.2f} minutes.")
            
        upper_outer = frame_data[frame_data['landmark_index'].isin(lips_upper_outer)]
        lower_outer = frame_data[frame_data['landmark_index'].isin(lips_lower_outer)]
        upper_inner = frame_data[frame_data['landmark_index'].isin(lips_upper_inner)]
        lower_inner = frame_data[frame_data['landmark_index'].isin(lips_lower_inner)]

        if not (upper_outer.empty or lower_outer.empty or upper_inner.empty or lower_inner.empty):
            # Horizontal distance (width of the smile)
            left_corner = frame_data[frame_data['landmark_index'] == lips_upper_outer[0]][['x_scaled', 'y_scaled']].values
            right_corner = frame_data[frame_data['landmark_index'] == lips_upper_outer[-1]][['x_scaled', 'y_scaled']].values
            if left_corner.size > 0 and right_corner.size > 0:
                horizontal_distance = np.linalg.norm(left_corner - right_corner)

            # Vertical distance (openness of the mouth)
            upper_mean = upper_inner[['x_scaled', 'y_scaled']].mean().values
            lower_mean = lower_inner[['x_scaled', 'y_scaled']].mean().values
            vertical_distance = np.linalg.norm(upper_mean - lower_mean)

            # Combined smile intensity
            smile_intensity = horizontal_distance + vertical_distance
            smile_intensities.append(smile_intensity)

    if smile_intensities:
        smile_metrics["average_smile_intensity"] = np.mean(smile_intensities)
        smile_metrics["max_smile_intensity"] = np.max(smile_intensities)

    print("Smiles calculated.")
    return smile_metrics


In [4]:
lips_upper_outer= [61, 185, 40, 39, 37, 0, 267, 269, 270, 409, 291]
lips_lower_outer= [146, 91, 181, 84, 17, 314, 405, 321, 375, 291]
lips_upper_inner= [78, 191, 80, 81, 82, 13, 312, 311, 310, 415, 308]
lips_lower_inner= [78, 95, 88, 178, 87, 14, 317, 402, 318, 324, 308]
smile_metrics = calculate_smiles(data, lips_upper_outer, lips_lower_outer, lips_upper_inner, lips_lower_inner)

Calculating smiles...
0 frames processed, corresponding to 0.00 minutes.
1620 frames processed, corresponding to 1.00 minutes.
3240 frames processed, corresponding to 2.00 minutes.
4860 frames processed, corresponding to 3.00 minutes.
6480 frames processed, corresponding to 4.00 minutes.
8100 frames processed, corresponding to 5.00 minutes.
9720 frames processed, corresponding to 6.00 minutes.
11340 frames processed, corresponding to 7.00 minutes.
12960 frames processed, corresponding to 8.00 minutes.
Smiles calculated.


In [40]:
smile_metrics

{'average_smile_intensity': 1.4963557863906305,
 'max_smile_intensity': 1.7086868985674104}

In [49]:
def calculate_head_movement(df, nose_tip_index):
    """
    Calculate head movement based on nose tip displacement in horizontal and vertical directions.

    Parameters:
        df (pd.DataFrame): DataFrame containing frame, landmark_index, x_scaled, y_scaled.
        nose_tip_index (int): Index of the nose tip landmark.

    Returns:
        dict: Contains total horizontal and vertical movements.
    """
    print("Calculating head movement...")
    head_metrics = {"total_horizontal_movement": 0, "total_vertical_movement": 0}
    nose_data = df[df['landmark_index'] == nose_tip_index].sort_values('frame')

    if not nose_data.empty:
        # Calculate frame-to-frame differences
        horizontal_diff = np.abs(np.diff(nose_data['x_scaled'], prepend=nose_data['x_scaled'].iloc[0]))
        vertical_diff = np.abs(np.diff(nose_data['y_scaled'], prepend=nose_data['y_scaled'].iloc[0]))

        # Sum up the movements
        head_metrics["total_horizontal_movement"] = np.sum(horizontal_diff)
        head_metrics["total_vertical_movement"] = np.sum(vertical_diff)

        print(f"Total horizontal movement: {head_metrics['total_horizontal_movement']:.4f}")
        print(f"Total vertical movement: {head_metrics['total_vertical_movement']:.4f}")

    print("Head movement calculation completed.")
    return head_metrics


In [55]:
head_metrics = calculate_head_movement(data, 1)

Calculating head movement...
Total horizontal movement: 109.4595
Total vertical movement: 79.6074
Head movement calculation completed.


In [56]:
head_metrics

{'total_horizontal_movement': 109.45951008188739,
 'total_vertical_movement': 79.60740305838294}

In [14]:
def calculate_yawns(df, upper_lip_indices, lower_lip_indices, yawn_threshold=0.1):
    """
    Calculate the number of yawns and the duration of yawns based on mouth aspect ratio.

    Parameters:
        df (pd.DataFrame): DataFrame containing frame, landmark_index, x_scaled, y_scaled.
        upper_lip_indices (list): List of indices for the upper lip.
        lower_lip_indices (list): List of indices for the lower lip.
        yawn_threshold (float): Threshold for detecting a yawn.

    Returns:
        dict: Contains yawn metrics like count, mean, max, and std duration.
    """
    print("Calculating yawns...")
    yawn_metrics = {
        "yawn_count": 0,
        "mean_yawn_duration": 0,
        "max_yawn_duration": 0,
        "std_yawn_duration": 0
    }

    def mouth_aspect_ratio(upper_lip_landmarks, lower_lip_landmarks):
        # Calculate the vertical distance between the upper and lower lip
        vertical_dist = np.linalg.norm(
            upper_lip_landmarks[['x_scaled', 'y_scaled']].mean().values -
            lower_lip_landmarks[['x_scaled', 'y_scaled']].mean().values
        )
        return vertical_dist
    
    # Combine upper and lower lip indices
    all_lip_indices = upper_lip_indices + lower_lip_indices
    
    # Filter for only lip landmarks
    lip_data = df[df['landmark_index'].isin(all_lip_indices)]
    
    yawn_timestamps = []
    yawn_durations = []
    is_yawning = False
    yawn_start = None
    
    for frame, frame_data in lip_data.groupby('frame'):
        if frame % (60 * 27) == 0:
            print(f"{frame} frames processed, corresponding to {frame / (60 * 27):.2f} minutes.")
            
        upper_lip = frame_data[frame_data['landmark_index'].isin(upper_lip_indices)]
        lower_lip = frame_data[frame_data['landmark_index'].isin(lower_lip_indices)]
        
        if not upper_lip.empty and not lower_lip.empty:
            mar = mouth_aspect_ratio(upper_lip, lower_lip)
            
            if mar > yawn_threshold:
                if not is_yawning:
                    is_yawning = True
                    yawn_start = frame
            else:
                if is_yawning:
                    is_yawning = False
                    yawn_durations.append(frame - yawn_start)
                    yawn_timestamps.append(frame/27)
    
    if yawn_durations:
        yawn_metrics["yawn_count"] = len(yawn_durations)
        yawn_metrics["mean_yawn_duration"] = np.mean(yawn_durations)
        yawn_metrics["max_yawn_duration"] = np.max(yawn_durations)
        yawn_metrics["std_yawn_duration"] = np.std(yawn_durations)
    
    print("Yawns calculated.")
    return yawn_metrics, yawn_durations, yawn_timestamps


In [17]:
upper_lip_indices = [61, 185, 40, 39, 37, 0, 267, 269, 270, 409, 291]  # Lips upper outer
lower_lip_indices = [146, 91, 181, 84, 17, 314, 405, 321, 375, 291]  # Lips lower outer
yawn_metrics, yawn_durations, yawn_timestamps = calculate_yawns(pd.read_csv("Data/face_landmarks/F1/Preprocessed/1_01_10.csv"), upper_lip_indices, lower_lip_indices, yawn_threshold=0.1)

Calculating yawns...
0 frames processed, corresponding to 0.00 minutes.
1620 frames processed, corresponding to 1.00 minutes.
3240 frames processed, corresponding to 2.00 minutes.
4860 frames processed, corresponding to 3.00 minutes.
6480 frames processed, corresponding to 4.00 minutes.
8100 frames processed, corresponding to 5.00 minutes.
9720 frames processed, corresponding to 6.00 minutes.
11340 frames processed, corresponding to 7.00 minutes.
12960 frames processed, corresponding to 8.00 minutes.
Yawns calculated.


In [19]:
yawn_metrics

{'yawn_count': 0,
 'mean_yawn_duration': 0,
 'max_yawn_duration': 0,
 'std_yawn_duration': 0}

In [8]:
def calculate_head_tilt(df, middle_line_indices):
    """
    Calculate head tilt based on the displacement of the middle line landmarks along the x and y axes.

    Parameters:
        df (pd.DataFrame): DataFrame containing frame, landmark_index, x_scaled, y_scaled.
        middle_line_indices (list): List of indices for the landmarks along the middle line of the face.

    Returns:
        dict: Contains mean, max, and std for x-axis and y-axis tilts.
    """
    print("Calculating head tilt...")
    tilt_metrics = {
        "y_tilt_mean": 0, "y_tilt_max": 0, "y_tilt_std": 0,
        "x_tilt_mean": 0, "x_tilt_max": 0, "x_tilt_std": 0,
    }

    middle_line_data = df[df['landmark_index'].isin(middle_line_indices)]

    if not middle_line_data.empty:
        # Group by frame
        tilts_y = []
        tilts_x = []
        for frame, frame_data in middle_line_data.groupby('frame'):
            y_coords = frame_data['y_scaled'].values
            x_coords = frame_data['x_scaled'].values

            # Calculate y-axis tilt (vertical spread)
            if len(y_coords) > 1:
                y_tilt = max(y_coords) - min(y_coords)
                tilts_y.append(y_tilt)

            # Calculate x-axis tilt (horizontal spread)
            if len(x_coords) > 1:
                x_tilt = max(x_coords) - min(x_coords)
                tilts_x.append(x_tilt)

        # Calculate metrics for y-axis tilt
        if tilts_y:
            tilt_metrics["y_tilt_mean"] = np.mean(tilts_y)
            tilt_metrics["y_tilt_max"] = np.max(tilts_y)
            tilt_metrics["y_tilt_std"] = np.std(tilts_y)

        # Calculate metrics for x-axis tilt
        if tilts_x:
            tilt_metrics["x_tilt_mean"] = np.mean(tilts_x)
            tilt_metrics["x_tilt_max"] = np.max(tilts_x)
            tilt_metrics["x_tilt_std"] = np.std(tilts_x)

    print("Head tilt calculated.")
    return tilt_metrics


In [9]:
calculate_head_tilt(data, )

Calculating head tilt...
Head tilt calculated.


{'y_tilt_mean': 4.592226048292161,
 'y_tilt_max': 4.816677614937163,
 'y_tilt_std': 0.03536946896574072,
 'x_tilt_mean': 0.37785153455562864,
 'x_tilt_max': 2.415456985988463,
 'x_tilt_std': 0.10292361733892635}